# Детекция фейковых новостей: Frozen LLM (ruGPT-3) + Sklearn Ensemble

**Ключевая идея**: используем настоящую **большую языковую модель** (ruGPT-3 small, 125M параметров) как замороженный семантический экстрактор, а классифицируем быстрыми sklearn-моделями.

### Почему ruGPT-3 — это LLM?

| Характеристика | ruGPT-3 small |
|----------------|---------------|
| Архитектура | GPT-2 (авторегрессионный трансформер) |
| Параметры | **125M** |
| Предобучение | Русскоязычный корпус (ai-forever) |
| Тип | **Decoder-only LLM** (генеративная модель) |

### Архитектура пайплайна

```
headline → frozen ruGPT-3 → mean_pool → h_emb (768)
body     → frozen ruGPT-3 → mean_pool → b_emb (768)
                                          ↓
              [h_emb, b_emb, |h-b|, h⊙b]  →  3072 признаков
                                          ↓
              LogReg / SVC / GradientBoosting → Soft Voting Ensemble
```

**Время работы**: ~5-10 мин (извлечение признаков) + секунды (обучение sklearn)

In [ ]:
import os, json, time, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix,
)

n_threads = os.cpu_count() or 4
torch.set_num_threads(n_threads)
torch.set_num_interop_threads(min(n_threads, 4))

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE} | CPU threads: {torch.get_num_threads()}')

In [ ]:
# ── Конфигурация ──────────────────────────────────────────────────────────────

DATA_PATH      = '../../data/ready_dataset.csv'
HEADLINE_COL   = 'headline_clean'
BODY_COL       = 'body_clean'
LABEL_COL      = 'label'

# Настоящая LLM: ruGPT-3 small (125M параметров, decoder-only GPT)
MODEL_NAME     = 'ai-forever/rugpt3small_based_on_gpt2'
MAX_LENGTH     = 64       # длина каждого сегмента (headline / body)
EXTRACT_BATCH  = 32       # батч для inference (GPT больше BERT, нужен меньший батч)
HIDDEN_SIZE    = 768      # ruGPT-3 small hidden dim

# Признаки: h_mean + b_mean + |h-b| + h*b = 4 × 768
FEAT_DIM       = HIDDEN_SIZE * 4

OUTPUT_DIR     = '../../models/llm_v2'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'LLM:      {MODEL_NAME}')
print(f'Params:   ~125M (decoder-only GPT)')
print(f'Feat dim: {FEAT_DIM}  (4 × {HIDDEN_SIZE})')

## 1. Загрузка данных

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df[[HEADLINE_COL, BODY_COL, LABEL_COL]].dropna()
df[HEADLINE_COL] = df[HEADLINE_COL].astype(str).str.strip()
df[BODY_COL]     = df[BODY_COL].astype(str).str.strip()
df = df[(df[HEADLINE_COL] != '') & (df[BODY_COL] != '')]
df[LABEL_COL]    = pd.to_numeric(df[LABEL_COL], errors='coerce').astype(int)

train_val, test_df = train_test_split(
    df, test_size=0.1, random_state=SEED, stratify=df[LABEL_COL],
)
train_df, val_df = train_test_split(
    train_val, test_size=0.1 / 0.9,
    random_state=SEED, stratify=train_val[LABEL_COL],
)
for d in (train_df, val_df, test_df):
    d.reset_index(drop=True, inplace=True)

print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')
print(f'Метки (train): {dict(train_df[LABEL_COL].value_counts())}')

## 2. Извлечение признаков из Frozen LLM (ruGPT-3)

### Почему frozen (замороженная) LLM?

| Подход | Время на CPU | Качество |
|--------|-------------|----------|
| Fine-tuning GPT (LoRA) | **Дни** | Хорошо |
| Frozen GPT + Sklearn | **Минуты** | Хорошо |

### Как работает экстракция?

GPT — **decoder-only** модель (нет сегментных эмбеддингов как у BERT), поэтому кодируем заголовок и тело **раздельно**:

```
headline → ruGPT-3 → hidden_states → mean_pool → h_emb (768 dims)
body     → ruGPT-3 → hidden_states → mean_pool → b_emb (768 dims)
```

Затем формируем **interaction features**:
- `h_emb` — семантика заголовка
- `b_emb` — семантика тела
- `|h_emb - b_emb|` — расхождение (фейковые заголовки расходятся с телом)
- `h_emb * b_emb` — поэлементное совпадение

Итого: **3072 признака** (4 × 768)

In [ ]:
class FrozenGPTExtractor:
    """
    Извлекает interaction features из замороженного ruGPT-3 small.

    GPT — decoder-only модель, поэтому headline и body кодируются раздельно.
    Выход: [h_mean | b_mean | |h-b| | h*b] = 4 × 768 = 3072 признака.
    """

    def __init__(self, model_name: str, max_length: int, device: str):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device)
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad = False
        self.max_length = max_length
        self.device = device

        # GPT-2 не имеет pad-токена по умолчанию
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        n_params = sum(p.numel() for p in self.model.parameters())
        h = self.model.config.hidden_size
        print(f'LLM: {model_name}')
        print(f'  hidden_size={h}  |  params={n_params:,}  |  FROZEN')
        print(f'  architecture: decoder-only GPT (autoregressive LLM)')

    @staticmethod
    def _mean_pool(hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """Маскированное среднее по временному измерению."""
        mf = mask.unsqueeze(-1).float()
        return (hidden * mf).sum(1) / mf.sum(1).clamp(min=1e-9)

    @torch.inference_mode()
    def _encode_texts(self, texts: list) -> torch.Tensor:
        """Кодирует список текстов → mean-pooled embeddings [B, H]."""
        enc = self.tokenizer(
            texts,
            truncation=True,
            max_length=self.max_length,
            padding=True,
            return_tensors='pt',
        )
        ids   = enc['input_ids'].to(self.device)
        masks = enc['attention_mask'].to(self.device)

        hidden = self.model(input_ids=ids, attention_mask=masks).last_hidden_state
        return self._mean_pool(hidden, masks)

    @torch.inference_mode()
    def _extract_batch(self, headlines: list, bodies: list) -> np.ndarray:
        """Извлекает interaction features для батча пар."""
        h_emb = self._encode_texts(headlines)  # [B, 768]
        b_emb = self._encode_texts(bodies)     # [B, 768]

        features = torch.cat([
            h_emb,
            b_emb,
            (h_emb - b_emb).abs(),
            h_emb * b_emb,
        ], dim=-1)  # [B, 4H]

        return features.cpu().numpy()

    def extract_all(self, headlines: list, bodies: list,
                    batch_size: int = 32) -> np.ndarray:
        """Извлекает признаки для всего датасета батчами."""
        parts = []
        n = len(headlines)
        for i in tqdm(range(0, n, batch_size), desc='Extracting (ruGPT-3)'):
            parts.append(self._extract_batch(
                headlines[i:i+batch_size],
                bodies[i:i+batch_size],
            ))
        return np.vstack(parts)


print('FrozenGPTExtractor готов')

## 3. Экстракция признаков (с кешированием)

Признаки из LLM извлекаются **один раз** и сохраняются на диск.
При повторном запуске загружаются мгновенно.

In [ ]:
cache_path = os.path.join(OUTPUT_DIR, 'gpt_features_cache.npz')

if os.path.exists(cache_path):
    print(f'Загружаем признаки из кеша: {cache_path}')
    cache = np.load(cache_path)
    train_feats  = cache['train_feats']
    val_feats    = cache['val_feats']
    test_feats   = cache['test_feats']
    train_labels = cache['train_labels']
    val_labels   = cache['val_labels']
    test_labels  = cache['test_labels']
    print(f'Загружено: train={train_feats.shape}, val={val_feats.shape}, test={test_feats.shape}')
else:
    print('Инициализируем frozen LLM (ruGPT-3 small)...')
    extractor = FrozenGPTExtractor(MODEL_NAME, MAX_LENGTH, DEVICE)

    t0 = time.time()
    print('\nЭкстракция признаков из LLM (один раз, потом кеш):')

    train_feats = extractor.extract_all(
        train_df[HEADLINE_COL].tolist(), train_df[BODY_COL].tolist(),
        batch_size=EXTRACT_BATCH,
    )
    val_feats = extractor.extract_all(
        val_df[HEADLINE_COL].tolist(), val_df[BODY_COL].tolist(),
        batch_size=EXTRACT_BATCH,
    )
    test_feats = extractor.extract_all(
        test_df[HEADLINE_COL].tolist(), test_df[BODY_COL].tolist(),
        batch_size=EXTRACT_BATCH,
    )
    train_labels = train_df[LABEL_COL].values
    val_labels   = val_df[LABEL_COL].values
    test_labels  = test_df[LABEL_COL].values

    np.savez_compressed(
        cache_path,
        train_feats=train_feats, val_feats=val_feats, test_feats=test_feats,
        train_labels=train_labels, val_labels=val_labels, test_labels=test_labels,
    )
    elapsed = time.time() - t0
    print(f'\nГотово за {elapsed:.1f}s ({elapsed/60:.1f} мин). Кеш: {cache_path}')
    del extractor
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f'Размер вектора признаков: {train_feats.shape[1]}')

## 4. Обучение sklearn-классификаторов на LLM-признаках

Три модели на признаках из ruGPT-3:

| Модель | Особенность |
|--------|-------------|
| **LogisticRegression** | Линейная модель, быстрая, устойчивая |
| **LinearSVC** | Линейный SVM, хорошо работает в высоком пространстве |
| **GradientBoosting** | Нелинейная модель, ловит сложные паттерны |

**Soft Voting Ensemble** — усреднение вероятностей всех трёх моделей.

Все классификаторы обучаются за **секунды** на готовых LLM-признаках.

In [ ]:
# ── Нормализация признаков ────────────────────────────────────────────────────

scaler = StandardScaler()
X_train = scaler.fit_transform(train_feats)
X_val   = scaler.transform(val_feats)
X_test  = scaler.transform(test_feats)

y_train = train_labels
y_val   = val_labels
y_test  = test_labels

# ── Определяем модели ────────────────────────────────────────────────────────

models = {
    'LogisticRegression': LogisticRegression(
        C=1.0, max_iter=1000, solver='lbfgs', random_state=SEED, n_jobs=-1,
    ),
    'LinearSVC': CalibratedClassifierCV(
        LinearSVC(C=1.0, max_iter=2000, random_state=SEED),
        cv=3,
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        subsample=0.8, random_state=SEED,
    ),
}

# ── Обучение ─────────────────────────────────────────────────────────────────

results = {}
trained_models = {}
t_total = time.time()

for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    val_preds = model.predict(X_val)
    val_acc = accuracy_score(y_val, val_preds)
    val_f1  = f1_score(y_val, val_preds, average='weighted')

    test_preds = model.predict(X_test)
    test_acc = accuracy_score(y_test, test_preds)
    test_f1  = f1_score(y_test, test_preds, average='weighted')

    results[name] = {
        'val_acc': val_acc, 'val_f1': val_f1,
        'test_acc': test_acc, 'test_f1': test_f1,
        'train_time': train_time,
    }
    trained_models[name] = model

    print(f'{name:25s} | val_acc={val_acc:.4f} | test_acc={test_acc:.4f} | '
          f'test_f1={test_f1:.4f} | {train_time:.1f}s')

print(f'\nВсе модели обучены за {time.time()-t_total:.1f}s')

## 5. Soft Voting Ensemble

Ансамбль усредняет **вероятности** всех трёх моделей.

In [ ]:
# ── Ensemble: усреднение вероятностей ─────────────────────────────────────────

ensemble_probs = np.zeros((len(X_test), 2))
for name, model in trained_models.items():
    ensemble_probs += model.predict_proba(X_test)
ensemble_probs /= len(trained_models)
ensemble_preds = ensemble_probs.argmax(axis=1)

ensemble_metrics = {
    'accuracy':  accuracy_score(y_test, ensemble_preds),
    'f1':        f1_score(y_test, ensemble_preds, average='weighted'),
    'precision': precision_score(y_test, ensemble_preds, average='weighted'),
    'recall':    recall_score(y_test, ensemble_preds, average='weighted'),
}

print('=' * 55)
print('ENSEMBLE (Soft Voting) — РЕЗУЛЬТАТЫ НА ТЕСТЕ')
print('=' * 55)
for k, v in ensemble_metrics.items():
    print(f'  {k:12s}: {v:.4f}')

# Лучшая одиночная модель по val_f1
best_single_name = max(results, key=lambda n: results[n]['val_f1'])
best_single_model = trained_models[best_single_name]

print(f'\nЛучшая одиночная: {best_single_name}')
print(f'  test_acc: {results[best_single_name]["test_acc"]:.4f}')
print(f'  test_f1:  {results[best_single_name]["test_f1"]:.4f}')

## 6. Результаты и анализ

In [ ]:
print(classification_report(
    y_test, ensemble_preds, target_names=['Фейк', 'Реальная'],
))

# ── Сравнение моделей V3 ──────────────────────────────────────────────────────

res_df = pd.DataFrame([
    {'Model': name, 'Val Acc': r['val_acc'], 'Test Acc': r['test_acc'],
     'Test F1': r['test_f1'], 'Train Time': f"{r['train_time']:.1f}s"}
    for name, r in results.items()
])
res_df.loc[len(res_df)] = {
    'Model': 'Ensemble (Soft Voting)',
    'Val Acc': '-',
    'Test Acc': ensemble_metrics['accuracy'],
    'Test F1': ensemble_metrics['f1'],
    'Train Time': '-',
}
print(res_df.to_string(index=False))

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────

cm = confusion_matrix(y_test, ensemble_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Фейк', 'Реальная'],
            yticklabels=['Фейк', 'Реальная'], ax=ax)
ax.set_title(
    f'Frozen ruGPT-3 + Sklearn Ensemble\n'
    f'Acc: {ensemble_metrics["accuracy"]:.4f}  F1: {ensemble_metrics["f1"]:.4f}',
    fontweight='bold',
)
ax.set_xlabel('Предсказано')
ax.set_ylabel('Истинно')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Сравнение со всеми моделями проекта

In [ ]:
comparison_data = []

if os.path.exists('../../models/llm/metrics.json'):
    with open('../../models/llm/metrics.json') as f:
        m = json.load(f)
        for key, name, method in [
            ('zero_shot_nli', 'Zero-shot NLI',  'LLM V1'),
            ('rugpt3_lora',   'ruGPT-3 + LoRA', 'LLM V1'),
        ]:
            if key in m:
                comparison_data.append({
                    'Model': name, 'Method': method,
                    'Accuracy': m[key]['accuracy'], 'F1': m[key]['f1'],
                    'Precision': m[key]['precision'], 'Recall': m[key]['recall'],
                })

if os.path.exists('../../models/llm_v2/metrics.json'):
    with open('../../models/llm_v2/metrics.json') as f:
        m = json.load(f)
        t = m.get('test', {})
        if t:
            comparison_data.append({
                'Model': 'Frozen BERT + MLP (V2)', 'Method': 'LLM V2',
                'Accuracy': t['accuracy'], 'F1': t['f1'],
                'Precision': t['precision'], 'Recall': t['recall'],
            })

if os.path.exists('../../models/rubert/metrics.json'):
    with open('../../models/rubert/metrics.json') as f:
        m = json.load(f)['rubert']
        comparison_data.append({
            'Model': 'RuBERT (full FT)', 'Method': 'Transformer',
            'Accuracy': m['test_acc'], 'F1': m['test_f1'],
            'Precision': m['test_precision'], 'Recall': m['test_recall'],
        })

if os.path.exists('../../results/metrics/metrics_tfidf_tuned.json'):
    with open('../../results/metrics/metrics_tfidf_tuned.json') as f:
        m = json.load(f)
        for name, key in [
            ('LR (TF-IDF)', 'logistic_regression'),
            ('NB (TF-IDF)', 'naive_bayes'),
            ('RF (TF-IDF)', 'random_forest'),
        ]:
            if key in m:
                comparison_data.append({
                    'Model': name, 'Method': 'Classical',
                    'Accuracy': m[key]['val_acc'], 'F1': m[key]['val_f1'],
                    'Precision': m[key].get('precision', 0),
                    'Recall': m[key].get('recall', 0),
                })

comparison_data.append({
    'Model': 'Frozen ruGPT-3 + Sklearn (V3)', 'Method': 'LLM V3',
    'Accuracy': ensemble_metrics['accuracy'], 'F1': ensemble_metrics['f1'],
    'Precision': ensemble_metrics['precision'], 'Recall': ensemble_metrics['recall'],
})

comparison_df = (
    pd.DataFrame(comparison_data)
    .sort_values('Accuracy', ascending=False)
    .reset_index(drop=True)
)
comparison_df.index = range(1, len(comparison_df) + 1)
print(comparison_df.to_string())
os.makedirs('assets', exist_ok=True)
comparison_df.to_csv('../../assets/all_models_comparison.csv', index=False)

In [ ]:
colors_map = {
    'Classical':   '#3498db',
    'Transformer': '#e74c3c',
    'LLM V1':      '#f39c12',
    'LLM V2':      '#27ae60',
    'LLM V3':      '#8e44ad',
}

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
for ax, metric in [
    (axes[0, 0], 'Accuracy'),
    (axes[0, 1], 'F1'),
    (axes[1, 0], 'Precision'),
    (axes[1, 1], 'Recall'),
]:
    plot_df = comparison_df.sort_values(metric, ascending=True)
    bars = ax.barh(
        plot_df['Model'], plot_df[metric],
        color=[colors_map.get(m, '#999') for m in plot_df['Method']],
        edgecolor='white', height=0.65,
    )
    ax.set_xlim(0.45, 1.02)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    for bar, val in zip(bars, plot_df[metric]):
        ax.text(val + 0.003, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=9, fontweight='bold')

from matplotlib.patches import Patch
fig.legend(
    handles=[Patch(facecolor=c, label=l) for l, c in colors_map.items()],
    loc='upper center', ncol=5, bbox_to_anchor=(0.5, 0.99),
)
fig.suptitle(
    'Сравнение всех подходов к детекции фейковых новостей',
    fontsize=15, fontweight='bold', y=1.01,
)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('../../assets/complete_models_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Сохранение артефактов

In [ ]:
# ── Сохраняем модели ─────────────────────────────────────────────────────────

joblib.dump(best_single_model, os.path.join(OUTPUT_DIR, 'best_classifier.pkl'))
joblib.dump(scaler, os.path.join(OUTPUT_DIR, 'scaler.pkl'))
joblib.dump(trained_models, os.path.join(OUTPUT_DIR, 'all_classifiers.pkl'))

# ── Метрики ──────────────────────────────────────────────────────────────────

with open(os.path.join(OUTPUT_DIR, 'metrics.json'), 'w', encoding='utf-8') as f:
    json.dump({
        'ensemble': ensemble_metrics,
        'individual': {name: {
            'test_acc': r['test_acc'], 'test_f1': r['test_f1'],
        } for name, r in results.items()},
        'best_single': best_single_name,
        'config': {
            'model':        MODEL_NAME,
            'architecture': 'Frozen ruGPT-3 (decoder-only LLM) + Sklearn Ensemble',
            'params':       '~125M',
            'max_length':   MAX_LENGTH,
            'feat_dim':     FEAT_DIM,
            'classifiers':  list(models.keys()),
        },
    }, f, indent=4, ensure_ascii=False)

# ── Предсказания на тесте ────────────────────────────────────────────────────

pred_df = test_df[[HEADLINE_COL, BODY_COL, LABEL_COL]].copy()
pred_df['pred']      = ensemble_preds
pred_df['prob_real'] = ensemble_probs[:, 1]
pred_df.to_csv(os.path.join(OUTPUT_DIR, 'test_predictions.csv'), index=False)

# ── Ошибки ───────────────────────────────────────────────────────────────────

errors = ensemble_preds != y_test
print(f'Правильных: {(~errors).sum()}/{len(y_test)}')
print(f'FP (фейк -> реальная): {((y_test==0) & (ensemble_preds==1)).sum()}')
print(f'FN (реальная -> фейк): {((y_test==1) & (ensemble_preds==0)).sum()}')

print('\nАртефакты сохранены:')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(fpath):
        print(f'  {fname}  ({os.path.getsize(fpath)/1024:.0f} KB)')